In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from util import read_classes, read_anchors
from model_torch import YOLOv2, train_model, wrap_yolo_loss, YoloV2GridTransform
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from dataset import VOCDataset, VOC_CLASSES
import matplotlib.pyplot as plt


In [ ]:
class_names = read_classes("model_data/coco_classes.txt")
print(class_names)
anchors = read_anchors("model_data/yolo_anchors.txt")
model_image_size = (608, 608) # Same as yolo_model input layer size
print(anchors)
num_classes = len(VOC_CLASSES)
num_anchors = len(anchors)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((416,416)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAutocontrast(0.1),
    v2.ToImage(),                          # 1. Converts PIL Image to a Tensor image
    v2.ToDtype(torch.float32, scale=True), # 2. Converts to Float32 AND sca# les values to [0.0, 1.0]
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Always last
])
generators = torch.Generator().manual_seed(42)
batch_size = 64

dataset = VOCDataset(
    root="model_data/VOC2007",
    split_file=(
        "model_data/VOC2007/"
        "ImageSets/Main/trainval.txt"
    ),
    batch_size=batch_size,
    grid_shape=(13,13),
    anchors=anchors,
    transform=data_transforms,
)

# train_dataset = datasets.ImageFolder("model_data/VOC2007", transform=data_transforms)
train_dataset, val_dataset = random_split(dataset, [.8, .2], generator=generators)
val_dataset, test_dataset = random_split(val_dataset, [.5, .5], generator=generators)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
image, target = train_dataset[0]
print(type(image), type(target))
print(image.size)
print(target)

In [ ]:
learning_rate = 1e-3
weight_decay = 1e-4
epochs = 3 # use small epochs for transfer learning

# model = CNNModel(num_classes).to(device)
model = YOLOv2(num_anchors, num_classes).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs, # number of epochs
    eta_min=1e-6
)

In [ ]:
# model = Darknet19()

# x = torch.randn(1, 3, 416, 416)
# with torch.no_grad():
#     y = model(x)

# print("output:", y.shape)

In [ ]:
epochs = 1
loss_fn = wrap_yolo_loss(loss_weight=[1,1,1,1])
history = train_model(epochs,model, train_dataloader, val_dataloader, loss_fn, optimizer,scheduler, batch_size, device, num_classes)

train_loop-0 <class 'torch.Tensor'> <class 'torch.Tensor'>
stage1: torch.Size([64, 32, 208, 208])
stage2: torch.Size([64, 64, 104, 104])
stage3: torch.Size([64, 128, 52, 52])
stage4: torch.Size([64, 256, 26, 26])
stage5: torch.Size([64, 512, 26, 26])
pool5: torch.Size([64, 512, 13, 13])
stage6: torch.Size([64, 1024, 13, 13])
before passthrough torch.Size([64, 512, 26, 26])
after passthrough torch.Size([64, 256, 13, 13])

x after concatenate		: torch.Size([64, 1280, 13, 13])
x after stage7 #0	: torch.Size([64, 1024, 13, 13])
x after stage7 #1	: torch.Size([64, 125, 13, 13])
torch.Size([64, 13, 13, 5]) torch.Size([64, 13, 13, 5, 25])
loss:     nan  [   64/ 4009]
train_loop-1 <class 'torch.Tensor'> <class 'torch.Tensor'>
stage1: torch.Size([64, 32, 208, 208])
stage2: torch.Size([64, 64, 104, 104])
stage3: torch.Size([64, 128, 52, 52])
stage4: torch.Size([64, 256, 26, 26])
stage5: torch.Size([64, 512, 26, 26])
pool5: torch.Size([64, 512, 13, 13])
stage6: torch.Size([64, 1024, 13, 13])
befo

In [ ]:
weights_path = "save/yolov2.pth"
torch.save(model.state_dict(), weights_path)
print(f"Weights successfully saved to {weights_path}")

In [ ]:
state_dict = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
print("Weights successfully loaded into model.")

In [ ]:
print("Precision:", history["val_precision"])
print("Recall:", history["val_recall"])
print("F1:", history["val_f1"])

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15,5))
plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()